In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

print("Environment Ready")

Environment Ready


In [3]:
df = pd.read_csv("heart_disease_uci.csv")
df.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [4]:
print("Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

Shape: (920, 16)

Missing Values:
 id            0
age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64

Duplicates: 0


In [15]:
df = df.drop(['id', 'dataset'], axis=1)
# no need for prediction


KeyError: "['id', 'dataset'] not found in axis"

In [14]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,63,Male,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,67,Male,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,67,Male,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,37,Male,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,41,Female,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [7]:
df.dtypes

age           int64
sex          object
cp           object
trestbps    float64
chol        float64
fbs          object
restecg      object
thalch      float64
exang        object
oldpeak     float64
slope        object
ca          float64
thal         object
num           int64
dtype: object

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
# Training before cleaning
# Temporary bad fill
df_temp = df.fillna(0)

# Encode categorical columns
df_encoded = pd.get_dummies(df_temp, drop_first=True)

# Use encoded dataframe
X = df_encoded.drop('num', axis=1)
y = df_encoded['num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

acc_before = accuracy_score(y_test, pred)
f1_before = f1_score(y_test, pred, average='weighted')

print("Before Cleaning")
print("Accuracy:", acc_before)
print("F1:", f1_before)

Before Cleaning
Accuracy: 0.5543478260869565
F1: 0.5028107966086443


In [9]:
df_cleaned = df.copy()

for col in df_cleaned.columns:
    if col != 'num':
        if df_cleaned[col].dtype == 'object':
            # Fill categorical with mode
            df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])
        else:
            # Fill numeric with median
            df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

C:\Users\srv00\AppData\Local\Temp\ipykernel_2620\1996187041.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])


In [10]:
df_cleaned_encoded = pd.get_dummies(df_cleaned, drop_first=True)

X2 = df_cleaned_encoded.drop('num', axis=1)
y2 = df_cleaned_encoded['num']

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

model2 = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42
)

model2.fit(X_train2, y_train2)

pred2 = model2.predict(X_test2)

acc_after = accuracy_score(y_test2, pred2)
f1_after = f1_score(y_test2, pred2, average='weighted')

print("After Cleaning")
print("Accuracy:", acc_after)
print("F1:", f1_after)

After Cleaning
Accuracy: 0.5652173913043478
F1: 0.5140136643875021


In [12]:
# using hyperparameter tuning
df_cleaned_encoded = pd.get_dummies(df_cleaned, drop_first=True)

X2 = df_cleaned_encoded.drop('num', axis=1)
y2 = df_cleaned_encoded['num']

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5]
}

rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy'
)

grid_search.fit(X_train2, y_train2)

best_model = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)

pred2 = best_model.predict(X_test2)

acc_tuned = accuracy_score(y_test2, pred2)
f1_tuned = f1_score(y_test2, pred2, average='weighted')

print("After Hyperparameter Tuning")
print("Accuracy:", acc_tuned)
print("F1:", f1_tuned)

Best Parameters: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 150}
After Hyperparameter Tuning
Accuracy: 0.5380434782608695
F1: 0.4537385955196128
